In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord

In [15]:
df = pd.read_csv('observaciones_astrometricas.csv')
df.iloc[24:48]

,object_key,object_name,object_type,horizons_id,datetime_utc,datetime_tdb,t_jd_tdb,ra_deg,dec_deg,rho_au,sigma_ra_arcsec,sigma_dec_arcsec,sigma_rho_au,earth_x_au,earth_y_au,earth_z_au
24,borisov,2I/Borisov,interestelar,2I,2019-11-24 23:58:50.817,A.D. 2019-Nov-25 00:00:00.0000,2458812.5,166.335149,-8.113784,2.104427,0.7,0.7,0.00006,0.461447,0.872774,-0.000043
25,borisov,2I/Borisov,interestelar,2I,2019-11-25 23:58:50.817,A.D. 2019-Nov-26 00:00:00.0000,2458813.5,166.765895,-8.846439,2.094349,0.7,0.7,0.00006,0.445881,0.880621,-0.000043
26,borisov,2I/Borisov,interestelar,2I,2019-11-26 23:58:50.817,A.D. 2019-Nov-27 00:00:00.0000,2458814.5,167.195700,-9.583268,2.084574,0.7,0.7,0.00006,0.430177,0.888195,-0.000043
27,borisov,2I/Borisov,interestelar,2I,2019-11-27 23:58:50.817,A.D. 2019-Nov-28 00:00:00.0000,2458815.5,167.624542,-10.324066,2.075104,0.7,0.7,0.00006,0.414339,0.895494,-0.000043
28,borisov,2I/Borisov,interestelar,2I,2019-11-28 23:58:50.817,A.D. 2019-Nov-29 00:00:00.0000,2458816.5,168.052401,-11.068626,2.065942,0.7,0.7,0.00006,0.398374,0.902516,-0.000042
29,borisov,2I/Borisov,interestelar,2I,2019-11-29 23:58:50.817,A.D. 2019-Nov-30 00:00:00.0000,2458817.5,168.479259,-11.816729,2.057090,0.7,0.7,0.00006,0.382287,0.909258,-0.000042
30,borisov,2I/Borisov,interestelar,2I,2019-11-30 23:58:50.817,A.D. 2019-Dec-01 00:00:00.0000,2458818.5,168.905096,-12.568151,2.048551,0.7,0.7,0.00006,0.366082,0.915718,-0.000042
31,borisov,2I/Borisov,interestelar,2I,2019-12-01 23:58:50.817,A.D. 2019-Dec-02 00:00:00.0000,2458819.5,169.329894,-13.322664,2.040325,0.7,0.7,0.00006,0.349765,0.921895,-0.000041
32,borisov,2I/Borisov,interestelar,2I,2019-12-02 23:58:50.817,A.D. 2019-Dec-03 00:00:00.0000,2458820.5,169.753635,-14.080029,2.032415,0.7,0.7,0.00006,0.333341,0.927785,-0.000041
33,borisov,2I/Borisov,interestelar,2I,2019-12-03 23:58:50.817,A.D. 2019-Dec-04 00:00:00.0000,2458821.5,170.176299,-14.840004,2.024822,0.7,0.7,0.00006,0.316815,0.933388,-0.000041


## ¿Qué tipo de cuerpos parecen estar incluidos en el archivo?
Parecen ser cometas, cuerpos con trayectorias hiperbólicas

## ¿Cómo se mueven sobre el cielo durante la ventana observada?
El que se escogió (Borisov) se mueve aumentando su ascención recta y disminuyendo su declinación, es decir, se mueve hacia el sur-oriente. La observación se realizó durante aproximadamente 20 días. Observando el dataframe, se notó que su distancia respecto a la tierra disminuyó aproximadamente 0.5 unidades astronómicas, lo cual es un valor considerable




Para conocer la velocidad, debido a que los datos están respecto a la tierra y, evidentemente, en un problema de dos cuerpos el cuerpo dominante es el sol, debemos pasar estos datos a coordenadas eclípticas.

In [38]:
earth2ecl = SkyCoord(ra = df['ra_deg'], dec = df['dec_deg'], unit = 'deg')\
    .transform_to('geocentrictrueecliptic') #transformamos a coordenadas eclípticas

#Ahora vamos a hallar las componentes cartesianas de las coordenadas eclípticas y centrar en el sol

x = earth2ecl.cartesian.x + df['earth_x_au']
y = earth2ecl.cartesian.y + df['earth_y_au']
z = earth2ecl.cartesian.z + df['earth_z_au']


Vamos a aproximar los datos del asteroide a un polinomio cuadrático, el cual describe de mejor manera el movimiento

In [ ]:
from scipy.optimize import curve_fit
from astropy.time import Time

tiempos = Time(df['t_jd'], format='jd').mjd - Time(df['t_jd'][0], format='jd').mjd
def cuadratica(t, a, b, c):
    return a*t**2 + b*t + c

t_datos = tiempos